In [7]:
import os
from tqdm import tqdm
from dotenv import load_dotenv

from llama_index.llms.openai import OpenAI
from llama_index.core.prompts import ChatMessage

from cores.distillation.generic_generation import generic_generate
from cores.utils import filter_query


load_dotenv()
llm = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-4o-mini", 
    temperature=1.0
) # set temp = 1.0


CATEGORY_EXAMPLE_MAPPING = {
    "Đồ uống": {
        "1 tỷ": [
            ["đi ăn nhà hàng sang trọng hết 1 tỏi",
            "mua bánh trung thu hết 2 tỏi rưỡi",
            "trà sữa cho cả công ty mất 3 tỏi 2",
            "đặt tiệc sinh nhật hết 7 tỏi",
            "gọi sushi thả ga hết một tỏi rưỡi",
            "đi ăn lẩu sang chảnh hết hai tỏi 3"],
            ["đi ăn nhà hàng sang trọng hết 1 tỏi 4 củ 10 cành",
            "mua trà sữa cho cả công ty mất 2 tỏi 3 triệu 5 trăm",
            "tổ chức tiệc tất niên hết 5 tỏi 2 củ 8 chục nghìn",
            "đi ăn buffet sang chảnh hết 3 tỏi 6 triệu 4 loét",
            "mua bánh trung thu cao cấp hết 1 tỏi 9 củ 7 lít",
            "đặt bàn tiệc sinh nhật hết 4 tỏi 5 triệu 6 cành",
            "đi nhậu cùng đối tác hết 2 tỏi 7 củ 9 loét",
            "gọi sushi cao cấp thả ga hết 6 tỏi 8 triệu 3 trăm",]
        ]
    },
    "Commute": {
    "1 tỷ": [
        [
            "đổ xăng cả năm hết 1 tỏi",
            "phí gửi xe chung cư hết 2 tỏi rưỡi",
            "bảo hiểm ô tô năm nay mất 3 tỏi 2",
            "thuê xe chạy hợp đồng hết 7 tỏi",
            "sửa xe sang chảnh hết một tỏi rưỡi",
            "mua gói bảo dưỡng định kỳ hai tỏi 3"
        ],
        [
            "bảo hiểm xe hơi mất 1 tỏi 4 củ 10 cành",
            "đổ xăng đường dài hết 2 tỏi 3 triệu 5 trăm",
            "sửa chữa xe hơi hết 5 tỏi 2 củ 8 chục nghìn",
            "đăng kiểm và bảo dưỡng ô tô hết 3 tỏi 6 triệu 4 loét",
            "phí gửi xe tòa nhà cao cấp hết 1 tỏi 9 củ 7 lít",
            "thuê xe đi công tác hết 4 tỏi 5 triệu 6 cành",
            "bảo hiểm xe sang mất 2 tỏi 7 củ 9 loét",
            "chi phí sửa xe tổng cộng 6 tỏi 8 triệu 3 trăm"
        ]
    ]
    },
        "Health_Care": {
        "1 tỷ": [
            [
                "phí khám sức khỏe tổng quát hết 1 tỏi",
                "mua thuốc bổ cho cả nhà hết 2 tỏi rưỡi",
                "đăng ký gói tập gym cao cấp mất 3 tỏi 2",
                "bảo hiểm y tế gia đình trọn gói hết 7 tỏi"
            ],
            [
                "phí phẫu thuật hết 1 tỏi 4 củ 10 cành",
                "mua thực phẩm chức năng hết 2 tỏi 3 triệu 5 trăm",
                "đăng ký gói tập yoga 5 tỏi 2 củ 8 chục nghìn",
                "bảo hiểm sức khỏe toàn diện hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    },
    "Living_Expense": {
        "1 tỷ": [
            [
                "tiền điện tháng này hết 1 tỏi",
                "hóa đơn tiền nước cả năm hết 2 tỏi rưỡi",
                "đóng tiền internet tốc độ cao mất 3 tỏi 2",
                "mua gas dự trữ cho cả năm hết 7 tỏi"
            ],
            [
                "phí truyền hình cáp hết 1 tỏi 4 củ 10 cành",
                "tiền điện thoại di động hết 2 tỏi 3 triệu 5 trăm",
                "tiền đi siêu thị tháng này hết 5 tỏi 2 củ 8 chục nghìn",
                "hóa đơn internet tốc độ cao hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    },

    "Child_Care": {
        "1 tỷ": [
            [
                "đóng học phí cho con hết 1 tỏi",
                "thuê người trông trẻ hết 2 tỏi rưỡi",
                "mua sữa cho bé mất 3 tỏi 2",
                "mua bỉm cho con cả năm hết 7 tỏi"
            ],
            [
                "mua đồ chơi trẻ em hết 1 tỏi 4 củ 10 cành",
                "tiền tiêu vặt cho con hết 2 tỏi 3 triệu 5 trăm",
                "mua xe đạp cho bé hết 5 tỏi 2 củ 8 chục nghìn",
                "đóng học phí trường quốc tế hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    },

    "Clothing": {
        "1 tỷ": [
            [
                "mua quần áo hàng hiệu hết 1 tỏi",
                "đầu tư giày sneaker sưu tầm hết 2 tỏi rưỡi",
                "mua phụ kiện thời trang mất 3 tỏi 2",
                "sắm đồ mùa đông cao cấp hết 7 tỏi"
            ],
            [
                "mua suit cao cấp hết 1 tỏi 4 củ 10 cành",
                "tậu túi hiệu hết 2 tỏi 3 triệu 5 trăm",
                "mua trang sức kim cương hết 5 tỏi 2 củ 8 chục nghìn",
                "mua giày da cao cấp hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    },

    "Household": {
        "1 tỷ": [
            [
                "mua nội thất cho nhà mới hết 1 tỏi",
                "tiền thuê nhà mỗi năm hết 2 tỏi rưỡi",
                "trả tiền thế chấp nhà mất 3 tỏi 2",
                "sửa chữa nhà cửa hết 7 tỏi"
            ],
            [
                "mua sofa nhập khẩu hết 1 tỏi 4 củ 10 cành",
                "đóng phí quản lý chung cư hết 2 tỏi 3 triệu 5 trăm",
                "nâng cấp phòng bếp hết 5 tỏi 2 củ 8 chục nghìn",
                "làm lại hệ thống điện hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    },

    "Treat_Money": {
        "1 tỷ": [
            [
                "đi vui chơi giải trí hết 1 tỏi",
                "chi tiền du lịch sang chảnh hết 2 tỏi rưỡi",
                "mua vé xem phim và ca nhạc mất 3 tỏi 2",
                "làm đẹp spa & massage hết 7 tỏi"
            ],
            [
                "mua mỹ phẩm cao cấp hết 1 tỏi 4 củ 10 cành",
                "du lịch Châu Âu hết 2 tỏi 3 triệu 5 trăm",
                "mua vé concert thần tượng hết 5 tỏi 2 củ 8 chục nghìn",
                "trải nghiệm dịch vụ VIP tại resort hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    },

    "Self_Growth": {
        "1 tỷ": [
            [
                "đầu tư học hành hết 1 tỏi",
                "chi tiền xây dựng mối quan hệ hết 2 tỏi rưỡi",
                "học khóa kỹ năng lãnh đạo mất 3 tỏi 2",
                "tham gia hội thảo doanh nhân hết 7 tỏi"
            ],
            [
                "học MBA hết 1 tỏi 4 củ 10 cành",
                "mua sách và tài liệu học tập hết 2 tỏi 3 triệu 5 trăm",
                "tham gia lớp coaching cá nhân hết 5 tỏi 2 củ 8 chục nghìn",
                "đăng ký khóa học startup hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    },

    "Bank": {
        "1 tỷ": [
            [
                "phí chuyển khoản quốc tế hết 1 tỏi",
                "trả lãi vay ngân hàng hết 2 tỏi rưỡi",
                "thanh toán nợ ngân hàng mất 3 tỏi 2",
                "mở tài khoản VIP tại ngân hàng hết 7 tỏi"
            ],
            [
                "phí duy trì tài khoản premium hết 1 tỏi 4 củ 10 cành",
                "đóng lãi vay bất động sản hết 2 tỏi 3 triệu 5 trăm",
                "trả nợ thẻ tín dụng hết 5 tỏi 2 củ 8 chục nghìn",
                "chuyển khoản đầu tư quốc tế hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    },

    "Invest": {
        "1 tỷ": [
            [
                "đầu tư chứng khoán hết 1 tỏi",
                "mua vàng tích trữ hết 2 tỏi rưỡi",
                "mua tiền số mất 3 tỏi 2",
                "mua trái phiếu dài hạn hết 7 tỏi"
            ],
            [
                "mua đất đầu tư hết 1 tỏi 4 củ 10 cành",
                "bỏ vốn startup hết 2 tỏi 3 triệu 5 trăm",
                "đầu tư quỹ mở hết 5 tỏi 2 củ 8 chục nghìn",
                "mua nhà chung cư để cho thuê hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    }
}

SYSTEM_MSG = ("You are a money manager assistant.\n"
                 "These under examples are sentences about spending money of value {value} VND for {subcategory}\n"
                 "Please generate 5 sentences that have value of {value} VND for {subcategory} similar to the examples.\n"
                 "EXAMPLES:\n{example}")
USER_MSG = "Similar sentences:\n"

SYSTEM_PROMPT = ChatMessage(
    role="system",
    content=SYSTEM_MSG
)
USER_PROMPT = ChatMessage(
    role="user",
    content=USER_MSG
)

for subcategory, value in CATEGORY_EXAMPLE_MAPPING.items():
    for money_value, samples in value.items():
        for i, sample in tqdm(enumerate(samples), desc=money_value):
            if sample: # tránh list rỗng
                example = "\n".join(sample)
                generated_sentences = ""
                for _ in range(5): # 50 câu
                    responses = generic_generate(
                        llm,
                        SYSTEM_PROMPT,
                        USER_PROMPT,
                        prompt_kwargs=dict(
                            value=money_value,
                            subcategory=subcategory,
                            example=example
                        )
                    )
                    responses = responses.split('\n')
                    for raw_response in responses:
                        if raw_response:
                            response = filter_query(raw_response)
                            generated_sentences += f"{response}\n"

                with open(f"data/generated/test_{subcategory}_{money_value}_{i}.txt", 'w', encoding="utf-8") as f:
                    f.write(generated_sentences)

1 tỷ: 2it [00:22, 11.29s/it]
1 tỷ: 2it [00:17,  8.71s/it]
1 tỷ: 2it [00:20, 10.02s/it]
1 tỷ: 2it [00:20, 10.33s/it]
1 tỷ: 2it [00:22, 11.36s/it]
1 tỷ: 2it [00:27, 13.99s/it]
1 tỷ: 2it [00:17,  8.78s/it]
1 tỷ: 2it [00:21, 10.92s/it]
1 tỷ: 2it [00:20, 10.32s/it]
1 tỷ: 2it [00:22, 11.32s/it]
1 tỷ: 2it [00:16,  8.37s/it]


In [8]:
import os

# Đường dẫn tới thư mục chứa các file
folder_path = "data/generated"

# Lấy danh sách file .txt trong thư mục, đảm bảo sắp xếp theo tên để tránh xáo trộn thứ tự
files = sorted([f for f in os.listdir(folder_path) if f.endswith(".txt")])

# Ghép nội dung từ tất cả các file, loại bỏ khoảng trắng dư thừa
all_content = []

for file in files:
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read().strip()  # Loại bỏ khoảng trắng đầu/cuối của từng file
        if content:  # Chỉ thêm nếu file có nội dung
            all_content.append(content)

# Ghi toàn bộ nội dung vào file mới, mỗi file cách nhau bởi dấu xuống dòng
output_file = "data/generated/merged.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(all_content))  # Ghép nội dung với xuống dòng giữa các file

print(f"Đã nối toàn bộ file vào {output_file} mà không có khoảng trắng dư thừa.")


Đã nối toàn bộ file vào data/generated/merged.txt mà không có khoảng trắng dư thừa.


You're an money manager assistant.
 Your job is to find and convert textual money string into integer money string

In [9]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
from tqdm import tqdm
from dotenv import load_dotenv

# from llama_index.llms.openai import OpenAI
from openai import OpenAI
from llama_index.core.prompts import ChatMessage

from cores.distillation.generic_generation import generic_generate
from cores.utils import filter_query


load_dotenv()
llm = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
) # set temp = 1.0

# Prompt to predict the money value of each sentence in file in data/generated/merged.txt
SYSTEM_PROMPT = """
You're an money manager assistant.
Your job is to find and convert textual money string into integer money string
Note that:
- The keywords ["triệu", 'm', "mê", "củ", "chai", "trai"] represent money with value of million
- The keywords ["trăm", "lít", "loét", "lốp", "lip", "líp", "list"] represent money with value of hundred thousand
- The keywords ["chục", "sịch", "xị", "sọi"] represent money with value of ten thousand
- The keywords ["k", "cành", "nghìn", "ngàn"] represent money with value of thousand
- The keywords ["tỷ", "tỉ", "tỏi"] represent money with value of billion
"""

USER_PROMPT = """
Only output the money value in integer, no other text
Input: {input}
"""

SYSTEM_PROMPT_XLSX = """
You're an money manager assistant.
Your job is to find and convert textual money string into integer money string
"""

# Read the file data/generated/merged.txt
with open("merged_bil.txt", "r", encoding="utf-8") as f:
    input_data = f.read()

# Predict the money value of each sentence in the file
responses = []
for sentence in input_data.split("\n"):
    value = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.replace("{input}", sentence)}
        ], 
        temperature=1.0
    ).choices[0].message.content
    responses.append({"system": SYSTEM_PROMPT_XLSX, "user": sentence, "value": value})


# Save to xlsx file with system columns is "You're an money manager assistant.Your job is to find and convert textual money string into integer money string", User is text that predict and value is output
import pandas as pd

df = pd.DataFrame(responses)
df.to_excel("bil_data.xlsx", index=False)
